# Mini NLP Pipeline

In [1]:
%pip install pandas fsspec huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_json("hf://datasets/dineshkarki/nepali-textbooks-corpus/data/dataset.jsonl", lines=True)

/home/nameless/workspace/ku/nlp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
corpus = df['text'].tolist()
corpus[0:10]

['सामाजिक अध्ययन तथा जीवनोपयोगी शिक्षा कक्षा १२ शिक्षा, विज्ञान तथा प्रविधि मन्त्रालय सानोठिमी, भक्तपुर प्रकाशक शिक्षा, विज्ञान तथा प्रविधि मन्त्रालय सानोठिमी, भक्तपुर (८ सर्वधिकार पाठ्यक्रम विकास केन्द्र यस पाठ्यपुस्तकसम्बन्धी सम्पूर्ण अधिकार पाठ्यक्रम विकास केन्द्र सानोठिमी, भक्तपुरमा निहित रहेको छ | पाठ्यक्रम विकास केन्द्रको लिखित स्वीकृतिबिना व्यापारिक प्रयोजनका लागि यसको पुरै वा आंशिक भाग हुबहु प्रकाशन गर्न, परिवर्तन गरेर प्रकाशन गर्न, कुनै विद्युतीय साधन वा अन्य प्रविधिबाट रेकर्ड गर्न र प्रतिलिपि निकाल्न पाइने छैन । पहिलो संस्करण : वि.सं २०७८ मुद्रण : जनक शिक्षा सामग्री केन्द्र लि. सानोठिमी, भक्तपुर । पाठयपुस्तकसस्बन्धी पाठकहरूका कुनै पति प्रकारका सझावहरू भएमा पाठ्यक्रम विकास Fez, ससमत्वय तथा प्रकाशव शाखामा पठाइदितुहुन अनुरोध छ | पाठकहरूबाट आउने सझावहरूलाई केन्द्र हार्दिक स्वागत गर्दछ । हाम्रो भनाइ शिक्षाले विद्यार्थीमा ज्ञानको खोजी गरी सिकाइ र वास्तविक जीवनबिच सम्बन्ध स्थापित गर्छ । शिक्षाले विद्यार्थीमा अधिकार, स्वतन्त्रता र समानताको प्रवर्धन गर्ने, स्वस्थ जीवनको अभ्यास गर्ने, 

# building vocab

In [4]:
import re
def tokenize(text):
    # No english words only devnagiri
    return re.findall(r'[\u0900-\u097F]+', text.lower())

In [5]:
all_tokens = [tokenize(text) for text in corpus]
unique_tokens = set(token for tokens in all_tokens for token in tokens)
vocab = list(unique_tokens - set(['', ' ', '\n', '\t']))
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")
print(f"First 10 tokens: {vocab[:10]}")

Vocabulary size: 113166
First 10 tokens: ['कामदार', 'सम्मान', 'गपप', 'सेवाबापत', 'जगत', 'सबसेलको', 'अस्थिरताको', 'गुटहरूको', 'अखबार', 'माइकबाट']


In [6]:
from collections import Counter, defaultdict
import math

tf_matrix = []
idf_dict = defaultdict(int)
total_docs = len(corpus)
for doc in corpus:
    tokens = tokenize(doc)
    token_counts = Counter(tokens)
    token_freq = {}
    for token in token_counts.keys():
        doc_len = len(doc)
        token_freq[token] = token_counts[token] / doc_len
        idf_dict[token] += 1
    tf_matrix.append(token_freq)

print("The term frequency for first document is :")
print(tf_matrix[0])
print("The most repeated words in the first document are :", 
      sorted(tf_matrix[0].items(), key=lambda x: x[1], reverse=True)[:10])
print("The inverse document frequency for each term is :")

tf_idf_matrix = []
for i, doc in enumerate(corpus):
    tokens = tokenize(doc)
    tf_idf_doc = defaultdict(float)
    for token in set(tokens):
        tf = tf_matrix[i].get(token, 0)
        idf = math.log(total_docs / idf_dict[token])
        tf_idf = tf * idf
        tf_idf_doc[token] = tf_idf
    tf_idf_matrix.append(tf_idf_doc)
print("The TF-IDF for first document is :")
print(sorted(tf_idf_matrix[1].items(), key=lambda x: x[1], reverse=True)[:10])

The term frequency for first document is :
{'सामाजिक': 0.0009290802105915144, 'अध्ययन': 0.0006193868070610096, 'तथा': 0.0021678538247135335, 'जीवनोपयोगी': 0.0006193868070610096, 'शिक्षा': 0.0018581604211830288, 'कक्षा': 0.0006193868070610096, '१२': 0.0006193868070610096, 'विज्ञान': 0.0009290802105915144, 'प्रविधि': 0.0009290802105915144, 'मन्त्रालय': 0.0009290802105915144, 'सानोठिमी': 0.0012387736141220192, 'भक्तपुर': 0.0009290802105915144, 'प्रकाशक': 0.0003096934035305048, '८': 0.0003096934035305048, 'सर्वधिकार': 0.0003096934035305048, 'पाठ्यक्रम': 0.001548467017652524, 'विकास': 0.002787240631774543, 'केन्द्र': 0.0021678538247135335, 'यस': 0.0021678538247135335, 'पाठ्यपुस्तकसम्बन्धी': 0.0003096934035305048, 'सम्पूर्ण': 0.0006193868070610096, 'अधिकार': 0.0006193868070610096, 'भक्तपुरमा': 0.0003096934035305048, 'निहित': 0.0003096934035305048, 'रहेको': 0.0006193868070610096, 'छ': 0.0021678538247135335, 'केन्द्रको': 0.0003096934035305048, 'लिखित': 0.0003096934035305048, 'स्वीकृतिबिना': 0.

In [7]:
def build_co_occurence_matrix(corpus, window_size=2):
    co_occurence_matrix = defaultdict(Counter)
    # Initialize the co-occurrence matrix with all tokens            
    for doc in corpus:
        tokens = tokenize(doc)
        for i, token in enumerate(tokens):
            # Define the window range
            start = max(0, i - window_size)
            end = min(len(tokens), i + window_size + 1)
            # Update co-occurrence counts for tokens within the window
            for j in range(start, end):
                if i != j:  # Exclude the token itself
                    co_occurence_matrix[token][tokens[j]] += 1
    return co_occurence_matrix

co_occurence_matrix = build_co_occurence_matrix(corpus, window_size=2)

# Print the co-occurrence matrix for the first 10 tokens
print("The co-occurrence matrix for the first 10 tokens is :")
for token in list(co_occurence_matrix.keys())[:10]:
    print(f"{token}: {dict(co_occurence_matrix[token])}")

The co-occurrence matrix for the first 10 tokens is :
सामाजिक: {'अध्ययन': 1584, 'तथा': 1490, 'आचरणको': 4, 'प्रदर्शन': 12, 'सद्भाव': 23, 'र': 868, '१२': 18, 'को': 18, 'पाठ': 45, '२': 65, 'अध्ययनमा': 8, 'सहसम्बन्ध': 2, '५': 24, 'अध्ययनका': 13, 'लागि': 58, '६': 18, 'तथ्याङ्क': 14, 'वर्तमान': 7, 'विश्वका': 2, 'विविधता': 28, 'सम्बन्धहरूको': 5, 'विकास': 143, 'अन्तरनिर्भरता': 14, '७९': 2, 'जीवन': 22, 'दर्शन': 6, 'मूल्य': 70, 'मान्यताहरू': 7, 'व्यवहार': 45, 'परिवर्तन': 26, 'रूपान्तरण': 23, '९६': 2, 'भूगोल': 7, '१०२': 2, 'नेपालको': 17, 'प्रदेशका': 3, 'सांस्कृतिक': 297, 'अवस्था': 36, 'जीवनमा': 11, 'सूचना': 9, '१०': 41, 'नक्साङ्कमा': 1, 'भूसूचना': 2, 'सदाचार': 5, 'जवाफदेही': 8, '२५८': 1, 'एकाइ': 15, '१': 66, 'जीवनोपयोगी': 5, 'शिक्षाको': 6, 'अवधारणा': 9, 'अन्तरसम्बन्ध': 9, 'अध्ययनअन्तर्गत': 2, 'मानव': 25, 'उसको': 9, 'भौतिक': 19, 'राजनीतिक': 115, 'गरिन्छ': 22, '।': 1079, 'नागरिक': 10, 'गरिने': 8, 'विज्ञानहरूको': 1, 'एकीकृत': 1, 'सिकाउँछ': 1, 'अध्ययनको': 6, 'पहिलो': 4, 'अन्तर्निर्भरता': 2, 'एवम्': 8

In [8]:
# Sparse entities in the whole co-occurrence matrix
sparsity = {token: 1 - len(co_occurence_matrix[token]) / vocab_size for token in co_occurence_matrix.keys()}

count_sparse = sum(1 for s in sparsity.values() if s > 0.9)
print(f"Number of sparse entities in the co-occurrence matrix: {count_sparse}")

Number of sparse entities in the co-occurrence matrix: 113163


In [9]:
token_counts = Counter()
total_tokens = 0

for doc in corpus:
    tokens = tokenize(doc)
    token_counts.update(tokens)
    total_tokens += len(tokens)

unigram_probabilities = {
    token: token_counts[token] / total_tokens
    for token in vocab
}

print("The unigram probabilities for the first 10 tokens are:")
for token in list(unigram_probabilities.keys())[:10]:
    print(f"{token}: {unigram_probabilities[token]}")

The unigram probabilities for the first 10 tokens are:
कामदार: 2.9544070761903048e-05
सम्मान: 0.00024191884029674235
गपप: 4.2817493857830504e-07
सेवाबापत: 1.7126997543132202e-06
जगत: 4.2817493857830504e-07
सबसेलको: 4.2817493857830504e-07
अस्थिरताको: 1.7126997543132202e-06
गुटहरूको: 4.2817493857830504e-07
अखबार: 8.135323832987795e-06
माइकबाट: 1.7126997543132202e-06


In [10]:
# Apply laplace smoothing to the unigram probabilities
laplace_smoothing = {
    token: (token_counts[token] + 1) / (total_tokens + vocab_size)
    for token in vocab
}

print("The Laplace smoothed unigram probabilities for the first 10 tokens are:")
for token in list(laplace_smoothing.keys())[:10]:
    print(f"{token}: {laplace_smoothing[token]}")


The Laplace smoothed unigram probabilities for the first 10 tokens are:
कामदार: 2.858706394517818e-05
सम्मान: 0.000231146831328155
गपप: 8.167732555765195e-07
सेवाबापत: 2.0419331389412984e-06
जगत: 8.167732555765195e-07
सबसेलको: 8.167732555765195e-07
अस्थिरताको: 2.0419331389412984e-06
गुटहरूको: 8.167732555765195e-07
अखबार: 8.167732555765194e-06
माइकबाट: 2.0419331389412984e-06


In [11]:
from collections import defaultdict, Counter

next_word_counts = defaultdict(Counter)

for doc in corpus:
    tokens = tokenize(doc)

    for i in range(len(tokens) - 1):
        current_word = tokens[i]
        next_word = tokens[i + 1]

        next_word_counts[current_word][next_word] += 1

In [12]:
while True:
    user_input = input("Enter a sentence (or type 'exit' to quit): ")

    if user_input.lower() == "exit":
        break

    input_tokens = tokenize(user_input)

    if not input_tokens:
        print("Please enter a sentence.")
        continue

    last_word = input_tokens[-1]

    if last_word in next_word_counts:
        candidates = next_word_counts[last_word]

        top_candidates = candidates.most_common(5)

        print("Top 5 next word predictions:")

        for word, count in top_candidates:
            print(f"{word}: {count}")
    else:
        print("No predictions available.")

Please enter a sentence.
Top 5 next word predictions:
वा: 20
र: 14
अमेरिकाको: 10
प्रमुख: 8
समयमा: 5
Top 5 next word predictions:
वा: 20
र: 14
अमेरिकाको: 10
प्रमुख: 8
समयमा: 5
Top 5 next word predictions:
वा: 20
र: 14
अमेरिकाको: 10
प्रमुख: 8
समयमा: 5
Top 5 next word predictions:
प्रहरी: 3
तथा: 1
स्तर: 1
गाउँपालिका: 1
कायम: 1
Top 5 next word predictions:
वहन: 22
पनि: 15
बोध: 14
पूरा: 12
तोकिएको: 9
Top 5 next word predictions:
के: 215
दिनुहोस्: 94
र: 73
लेख्नुहोस्: 50
बन्न: 37


KeyboardInterrupt: Interrupted by user